# APIs Lab

In this lab, you will work with different APIs to build datasets and perform basic analysis.

The goal is to practice how to:

- Make API requests  
- Explore JSON responses  
- Extract relevant information  
- Transform data into pandas DataFrames  
- Generate simple insights  

You will work like a data analyst using APIs data.

# Challenge 1: Pokémon Dataset & Analysis

## Goal

Build and analyze a Pokémon dataset using the API.

## Example (single request)

Below you can see how to get data for ONE Pokémon:

In [1]:
import requests
import pandas as pd

url = "https://pokeapi.co/api/v2/pokemon/1"

response = requests.get(url)
data = response.json()

data

{'id': 1,
 'name': 'bulbasaur',
 'base_experience': 64,
 'height': 7,
 'is_default': True,
 'order': 1,
 'weight': 69,
 'abilities': [{'is_hidden': False,
   'slot': 1,
   'ability': {'name': 'overgrow',
    'url': 'https://pokeapi.co/api/v2/ability/65/'}},
  {'is_hidden': True,
   'slot': 3,
   'ability': {'name': 'chlorophyll',
    'url': 'https://pokeapi.co/api/v2/ability/34/'}}],
 'past_abilities': [{'generation': {'name': 'generation-iv',
    'url': 'https://pokeapi.co/api/v2/generation/4/'},
   'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}],
 'forms': [{'name': 'bulbasaur',
   'url': 'https://pokeapi.co/api/v2/pokemon-form/1/'}],
 'game_indices': [{'game_index': 153,
   'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}},
  {'game_index': 153,
   'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}},
  {'game_index': 153,
   'version': {'name': 'yellow',
    'url': 'https://pokeapi.co/api/v2/version/3/'}},
  {'game_in

## Tasks

1. Using the example above, get data for Pokémon IDs from 1 to 151

2. Extract:
   - name
   - height
   - weight
   - base_experience
   - first type

3. Create a DataFrame

4. Add a new column:
   - weight_kg (divide weight by 10)

5. Answer:
   - Which Pokémon is the heaviest?
   - What is the average weight?
   - How many Pokémon are there per type?

6. Answer:
   - Which type has the highest average weight?

In [2]:

pokemon_list = []

for pokemon_id in range(1, 152):
    url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        name = data["name"]
        height = data["height"]
        weight = data["weight"]
        base_experience = data["base_experience"]
        type = data["types"][0]["type"]["name"]

        pokemon_list.append({
            "name": name,
            "height": height,
            "weight": weight,
            "base_experience": base_experience,
            "type": type

        })

In [3]:
pokemon = pd.DataFrame(pokemon_list)
pokemon

,name,height,weight,base_experience,type
0,bulbasaur,7,69,64,grass
1,ivysaur,10,130,142,grass
2,venusaur,20,1000,236,grass
3,charmander,6,85,62,fire
4,charmeleon,11,190,142,fire
...,...,...,...,...,...
146,dratini,18,33,60,dragon
147,dragonair,40,165,147,dragon
148,dragonite,22,2100,270,dragon
149,mewtwo,20,1220,306,psychic


In [4]:
pokemon["weight_kg"] = pokemon["weight"] / 10
pokemon

,name,height,weight,base_experience,type,weight_kg
0,bulbasaur,7,69,64,grass,6.9
1,ivysaur,10,130,142,grass,13.0
2,venusaur,20,1000,236,grass,100.0
3,charmander,6,85,62,fire,8.5
4,charmeleon,11,190,142,fire,19.0
...,...,...,...,...,...,...
146,dratini,18,33,60,dragon,3.3
147,dragonair,40,165,147,dragon,16.5
148,dragonite,22,2100,270,dragon,210.0
149,mewtwo,20,1220,306,psychic,122.0


In [5]:
heaviest_pokemon = pokemon.loc[pokemon["weight"].idxmax(), "name"]
print(heaviest_pokemon)

snorlax


In [6]:
average_weight = pokemon["weight"].mean()
print(average_weight)

459.5165562913907


In [7]:
total_per_type = pokemon["type"].value_counts()
print(total_per_type)

type
water       28
normal      22
poison      14
grass       12
fire        12
bug         12
electric     9
rock         9
ground       8
psychic      8
fighting     7
ghost        3
dragon       3
fairy        2
ice          2
Name: count, dtype: int64


In [8]:
weight_per_type = pokemon.groupby("type")["weight"].mean().idxmax()
print(weight_per_type)

rock


# Challenge 2: Crypto Market Analysis

## Goal

Analyze cryptocurrency prices using real-time data.

## Tasks

1. Get prices for:
   - bitcoin
   - ethereum
   - solana
   - dogecoin

2. Convert the response into a DataFrame (coins as rows)

3. Rename columns properly

4. Add:
   - price_rank (ranking by price)
   - price_category:
        - "high" if > 1000  
        - "medium" if between 100 and 1000  
        - "low" if < 100  

5. Answer:
   - Which coin is the most expensive?
   - Which category has more coins?

6. **Bonus**:
   - Create a column with price difference vs bitcoin

In [9]:
# starter code

import requests
import pandas as pd

url = "https://api.coingecko.com/api/v3/simple/price"

params = {
    "ids": "bitcoin,ethereum,solana,dogecoin",
    "vs_currencies": "usd"
}

response = requests.get(url, params=params)
data = response.json()

crypto = pd.DataFrame(data).T
print(crypto)

                   usd
bitcoin   77526.000000
ethereum   2249.860000
solana       95.940000
dogecoin      0.075707


In [10]:
crypto = crypto.rename(columns={"usd": "price(usd)"})

crypto

,price(usd)
bitcoin,77526.000000
ethereum,2249.860000
solana,95.940000
dogecoin,0.075707


In [11]:
crypto["price_rank"] = crypto["price(usd)"].rank(ascending=False, method="min").astype(int)

crypto

,price(usd),price_rank
bitcoin,77526.000000,1
ethereum,2249.860000,2
solana,95.940000,3
dogecoin,0.075707,4


In [13]:
crypto["price_category"] = "Medium"

crypto.loc[crypto["price(usd)"] < 100, "price_category"] = "Low"

crypto.loc[crypto["price(usd)"] > 1000, "price_category"] = "High"

crypto

,price(usd),price_rank,price_category
bitcoin,77526.000000,1,High
ethereum,2249.860000,2,High
solana,95.940000,3,Low
dogecoin,0.075707,4,Low


In [14]:
"""The most expensive coin is Bitcoin and in terms of category with the most coins "High" and "Low" have the same amount and "Medium" has none"""

'The most expensive coin is Bitcoin and in terms of category with the most coins "High" and "Low" have the same amount and "Medium" has none'